In [1]:
import sys
import argparse
import os
import numpy as np
import anndata
import statsmodels.api as sm
import statsmodels.formula.api as smf
import tqdm
import pandas as pd



import anndata
import csv
import gzip
import os
import scipy.io
import numpy as np 
import anndata
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import scanpy
from matplotlib.pyplot import rc_context




In [2]:

# --- INIT ---

# which data to use
head_folder = '/Users/benauerbach/Dropbox/aorta_circadian_data/datasets/joint'
hvg_to_use = 'transformed_X_outlier_variance'
cluster_resolution = 0.05

# get corresponding paths
adata_path = '%s/adata_qc_filtered.h5ad' % head_folder
hvg_folder = '%s/data_annotations/hvg/%s' % (head_folder,hvg_to_use)
scvi_res_folder = '%s/scvi_res' % hvg_folder
scvi_mean_embedding_df_path = '%s/scvi_mean_embedding.tsv' % scvi_res_folder
umap_path_out = '%s/scvi_mean_umap_embedding.tsv' % scvi_res_folder
clustering_folder = '%s/clustering/res_%s' % (scvi_res_folder, str(cluster_resolution))
cluster_df_fileout = '%s/clusters.tsv' % (clustering_folder)





In [3]:
# --- LOAD ADATA ---

adata = anndata.read_h5ad(adata_path)



In [4]:
# --- ADD EMBEDDINGS TO ADATA ---



# ** load embeddings and clusters **
scvi_mean_embedding_df = pd.read_table(scvi_mean_embedding_df_path,sep='\t',index_col='barcode')
umap_embedding_df = pd.read_table(umap_path_out,sep='\t',index_col='barcode')
cluster_df = pd.read_table(cluster_df_fileout,sep='\t',index_col='index')

# ** make sure everything in the same order
adata = adata[list(scvi_mean_embedding_df.index)]
umap_umap_embedding_df = umap_embedding_df.loc[list(scvi_mean_embedding_df.index)]
cluster_df = cluster_df.loc[list(scvi_mean_embedding_df.index)]


# ** add embeddings **
adata.obsm["X_scVI"] = np.array(scvi_mean_embedding_df)
adata.obsm["X_scVI_umap"] = np.array(umap_embedding_df)
adata.obs["cluster"] = np.array(cluster_df['leiden_scvi_cluster'])


adata


/opt/miniconda3/envs/tempo/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'
    obsm: 'X_scVI', 'X_scVI_umap'

In [5]:
# --- GET THE UNIQUE DESCRIPTION AND CLUSTERS ---

descriptions = list(adata.obs['description'].unique())
clusters = list(adata.obs['cluster'].unique())

print("Unique descriptions:\n",descriptions)
print("Unique clusters:\n",clusters)




Unique descriptions:
 ['male aligned bmal1-ko', 'male misaligned bmal1-control', 'female aligned bmal1-ko', 'female misaligned bmal1-control', 'male aligned bmal1-control', 'female aligned bmal1-control']
Unique clusters:
 [0, 1, 2, 3, 4, 6, 5]


In [6]:
# --- GET THE FOLDER OUTS ---

# head folder
reg_head_folder = '%s/point_est_harmonic_reg' % clustering_folder


# subfolders for each cluster / condition combo
cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (reg_head_folder,cluster)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        # update dict
        if cluster not in cluster_description_folder_out_dict:
            cluster_description_folder_out_dict[cluster] = {}
        cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder


    

In [29]:
# --- GET CLUSTER AND DESCRIPTION --

cluster = 1
description = 'male aligned bmal1-control'

# ** get cluster, condition adata **
cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
# cluster_condition_adata = cluster_condition_adata[:,genes_to_est]

cluster_condition_adata



/opt/miniconda3/envs/tempo/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)


View of AnnData object with n_obs × n_vars = 4194 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'
    obsm: 'X_scVI', 'X_scVI_umap'

In [30]:
# get the fileout
folder_out = cluster_description_folder_out_dict[cluster][description]
fileout = '%s/gene_param_df.tsv' % folder_out

# load the parameter df
gene_df = pd.read_table(fileout,sep='\t',index_col='gene')

# limit adata to the genes estimated
cluster_condition_adata = cluster_condition_adata[:,gene_df.index]
cluster_condition_adata


View of AnnData object with n_obs × n_vars = 4194 × 12111
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'
    obsm: 'X_scVI', 'X_scVI_umap'

In [31]:
# get the gene parameters
mesor = np.array(gene_df['mesor'])
amp = np.array(gene_df['amp'])
shift = np.array(gene_df['acrophase'])
theta = np.array(cluster_condition_adata.obs['zt'] / 24.0) * np.pi
log_L = np.log(np.array(cluster_condition_adata.obs['lib_size']))


# get the X
try:
    X = np.array(cluster_condition_adata.X.todense())
except:
    X = np.array(cluster_condition_adata.X)
    
# compute the LL under the sinusoid: [cells x genes]
log_prop = mesor + (amp * np.cos(theta.reshape(-1,1) - shift))
log_mean = log_prop + log_L.reshape(-1,1)
cell_gene_sinusoid_ll = scipy.stats.poisson(np.exp(log_mean)).logpmf(X)

# compute the LL under the flat: [cells x genes]
log_prop = mesor.reshape(1,-1)
log_mean = log_prop + log_L.reshape(-1,1)
cell_gene_flat_ll = scipy.stats.poisson(np.exp(log_mean)).logpmf(X)

# get gene LL
gene_sinusoid_ll = np.sum(cell_gene_sinusoid_ll,axis=0)
gene_flat_ll = np.sum(cell_gene_flat_ll,axis=0)

# get gene LRT
gene_log_lrt = gene_sinusoid_ll - gene_flat_ll
gene_lrt = np.exp(gene_log_lrt)
cluster_condition_adata.var['lrt'] = gene_lrt
cluster_condition_adata.var['sinusoid_ll'] = gene_lrt




Trying to set attribute `.var` of view, copying.


In [33]:
cluster_condition_adata.var['lrt'].sort_values(ascending=False).loc['Dbp']




0.0

In [16]:
gene_flat_ll = np.sum(cell_gene_flat_ll,axis=0)



In [17]:
gene_flat_ll

array([ -157.05318135, -2769.29968577,  -106.36848568, ...,
        -120.24779713,  -423.71589489,  -419.92859077])